# Frequenzimetro TXPRGDIVCLK — KR260 SFP+ SDM

Misura la frequenza di `TXPRGDIVCLK` (TXOUTCLK del GTHE4_CHANNEL via BUFG_GT),
che è derivata dalla QPLL0. Variando la parola SDM via DRP COMMON si osserva
la variazione di frequenza in tempo reale.

## Mappa AXI
| Base | Periferica |
|------|------------|
| 0xA0000000 | DRP GTHE4_CHANNEL |
| 0xA0010000 | AXI IIC SFP+ |
| 0xA0020000 | DRP GTHE4_COMMON (SDM/QPLL) |
| 0xA0030000 | **Frequenzimetro** |

## Registri frequenzimetro (offset da 0xA0030000)
| Offset | Nome | Accesso | Descrizione |
|--------|------|---------|-------------|
| 0x00 | GATE_CYCLES | r/w | Finestra di misura in cicli pl_clk0 (default 100 000 000 = 1 s @100 MHz) |
| 0x04 | FREQ_CNT | r/o | Spigoli contati di TXPRGDIVCLK nell'ultima finestra |
| 0x08 | STATUS | r/o | bit[0]=1 → nuovo risultato; si azzera leggendo STATUS |

## Stima della frequenza QPLL0 VCO
Per TXOUT_DIV=1 e TX_DATA_WIDTH=64 (TXPRGDIVCLK ≈ line_rate / 64):
```
QPLL0_VCO_MHz ≈ FREQ_CNT × 64 / gate_seconds
```
Se si usa il GTH Wizard con parametri diversi, aggiornare il moltiplicatore.

In [ ]:
import mmap
import struct
import time

# Indirizzi fisici AXI
FC_BASE   = 0xA003_0000
COMM_BASE = 0xA002_0000   # DRP GTHE4_COMMON
PAGE_SIZE = 0x1000

# Offset registri frequenzimetro
GATE_OFF = 0x00
CNT_OFF  = 0x04
STAT_OFF = 0x08

# Offset registri DRP bridge (CHANNEL e COMMON)
DRP_ADDR_OFF = 0x00
DRP_DI_OFF   = 0x04
DRP_DO_OFF   = 0x08
DRP_EN_OFF   = 0x0C
DRP_WE_OFF   = 0x10
DRP_RDY_OFF  = 0x14

def open_axi(base, size=PAGE_SIZE):
    """Apre /dev/mem e mappa la regione AXI specificata."""
    f = open('/dev/mem', 'r+b')
    mm = mmap.mmap(f.fileno(), size, offset=base)
    return f, mm

def rd32(mm, offset):
    mm.seek(offset)
    return struct.unpack('<I', mm.read(4))[0]

def wr32(mm, offset, value):
    mm.seek(offset)
    mm.write(struct.pack('<I', value & 0xFFFFFFFF))

def drp_read(mm, addr):
    """Legge un registro DRP (CHANNEL o COMMON)."""
    wr32(mm, DRP_ADDR_OFF, addr)
    wr32(mm, DRP_WE_OFF,   0)
    wr32(mm, DRP_EN_OFF,   1)
    timeout = 1000
    while timeout > 0:
        if rd32(mm, DRP_RDY_OFF) & 1:
            return rd32(mm, DRP_DO_OFF) & 0xFFFF
        time.sleep(0.001)
        timeout -= 1
    raise TimeoutError(f'DRP read timeout addr=0x{addr:02X}')

def drp_write(mm, addr, data):
    """Scrive un registro DRP (CHANNEL o COMMON)."""
    wr32(mm, DRP_ADDR_OFF, addr)
    wr32(mm, DRP_DI_OFF,   data & 0xFFFF)
    wr32(mm, DRP_WE_OFF,   1)
    wr32(mm, DRP_EN_OFF,   1)
    timeout = 1000
    while timeout > 0:
        if rd32(mm, DRP_RDY_OFF) & 1:
            return
        time.sleep(0.001)
        timeout -= 1
    raise TimeoutError(f'DRP write timeout addr=0x{addr:02X}')

print('Helper functions caricate.')

In [ ]:
# --- Configurazione finestra di misura ---
# gate_seconds: durata di ogni misura (es. 2.0 = 2 secondi)
gate_seconds = 2.0
pl_clk0_hz   = 100_000_000   # pl_clk0 nominale; verificare con get_property CONFIG.PSU__CRL_APB__PL0_REF_CTRL__FREQMHZ
gate_cycles  = int(gate_seconds * pl_clk0_hz)

# Moltiplicatore per stimare VCO da TXPRGDIVCLK
# Per TX_DATA_WIDTH=64, TXOUT_DIV=1: TXPRGDIVCLK = VCO / 64
# Aggiornare se si cambia la configurazione GTH
PROGDIV_RATIO = 64

f_fc, mm_fc = open_axi(FC_BASE)
wr32(mm_fc, GATE_OFF, gate_cycles)
print(f'Finestra impostata a {gate_seconds} s ({gate_cycles} cicli @ {pl_clk0_hz/1e6:.0f} MHz)')
print(f'Prima misura disponibile dopo ~{gate_seconds:.1f} s ...')

In [ ]:
# --- Singola lettura frequenza ---
def read_frequency(mm, timeout_s=10.0):
    """Attende il prossimo risultato e ritorna (freq_txprgdiv_hz, freq_vco_hz)."""
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if rd32(mm, STAT_OFF) & 1:
            cnt = rd32(mm, CNT_OFF)
            freq_tx = cnt / gate_seconds
            freq_vco = freq_tx * PROGDIV_RATIO
            return freq_tx, freq_vco
        time.sleep(0.05)
    raise TimeoutError('Nessun risultato entro il timeout')

freq_tx, freq_vco = read_frequency(mm_fc)
print(f'TXPRGDIVCLK = {freq_tx/1e6:.4f} MHz')
print(f'QPLL0 VCO stima = {freq_vco/1e9:.6f} GHz')
print(f'  (atteso: 10.3125 GHz con FBDIV=66, SDM_DATA=0)')

In [ ]:
# --- Modifica SDM via DRP COMMON e osserva variazione di frequenza ---
# Registri QPLL0_SDM (UG576 v1.7.1, Table C-1, GTHE4_COMMON DRP Map):
#   QPLL0_SDM_CFG0 @ DRP addr 0x0020
#     bit[14]=0  -> SDM abilitato
#     bits[12:0] = SDM_DATA[12:0]  (13 bit bassi della parola frazionaria 22-bit)
#   QPLL0_SDM_CFG1 @ DRP addr 0x0021
#     bits[8:0]  = SDM_DATA[21:13]  (9 bit alti)
#   QPLL0_SDM_CFG2 @ DRP addr 0x0024  (lasciar invariato per ora)
#
# Esempio: offset di +1 LSB sulla parola SDM da 22 bit
#   VCO shift = Fref / 2^22 = 156.25 MHz / 4194304 = 37.3 Hz (minimo step)

SDM_CFG0_ADDR = 0x0020   # QPLL0_SDM_CFG0 (UG576 v1.7.1)
SDM_CFG1_ADDR = 0x0021   # QPLL0_SDM_CFG1 (UG576 v1.7.1)
SDM_CFG2_ADDR = 0x0024   # QPLL0_SDM_CFG2 (UG576 v1.7.1)

f_comm, mm_comm = open_axi(COMM_BASE)

def set_sdm_data(mm_common, sdm_data_22bit):
    """Scrive la parola frazionaria SDM a 22 bit nella QPLL0."""
    lo13 = sdm_data_22bit & 0x1FFF         # bit[12:0]
    hi9  = (sdm_data_22bit >> 13) & 0x1FF  # bit[21:13]
    cfg0 = lo13          # bit[14]=0 -> SDM abilitato
    cfg1 = hi9
    drp_write(mm_common, SDM_CFG0_ADDR, cfg0)
    drp_write(mm_common, SDM_CFG1_ADDR, cfg1)
    print(f'SDM_DATA=0x{sdm_data_22bit:06X}  CFG0=0x{cfg0:04X}  CFG1=0x{cfg1:04X}')

# Leggi configurazione attuale
cfg0_cur = drp_read(mm_comm, SDM_CFG0_ADDR)
cfg1_cur = drp_read(mm_comm, SDM_CFG1_ADDR)
sdm_cur  = ((cfg1_cur & 0x1FF) << 13) | (cfg0_cur & 0x1FFF)
print(f'SDM_DATA corrente: 0x{sdm_cur:06X} ({sdm_cur})')
print(f'  CFG0=0x{cfg0_cur:04X}  CFG1=0x{cfg1_cur:04X}')

In [ ]:
# --- Sweep SDM e grafico frequenza ---
import matplotlib.pyplot as plt
import numpy as np

# Sweep lineare: SDM_DATA da 0 a 2^22-1 in N passi
N_STEPS     = 20
SDM_MAX     = 2**22 - 1
sdm_values  = np.linspace(0, SDM_MAX // 4, N_STEPS, dtype=int)  # primo quarto per demo

freqs_tx  = []
freqs_vco = []
sdm_used  = []

for sdm_val in sdm_values:
    set_sdm_data(mm_comm, int(sdm_val))
    time.sleep(0.1)   # tempo per QPLL re-lock
    try:
        ft, fv = read_frequency(mm_fc, timeout_s=gate_seconds + 3)
        freqs_tx.append(ft / 1e6)
        freqs_vco.append(fv / 1e9)
        sdm_used.append(int(sdm_val))
        print(f'SDM=0x{int(sdm_val):06X}  TX={ft/1e6:.4f} MHz  VCO≈{fv/1e9:.6f} GHz')
    except TimeoutError:
        print(f'SDM=0x{int(sdm_val):06X}  TIMEOUT (QPLL non locked?)')

# Ripristina SDM_DATA = 0
set_sdm_data(mm_comm, 0)
print('\nSDM_DATA ripristinato a 0')

# --- Grafico ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(sdm_used, freqs_tx, 'o-', color='steelblue')
ax1.set_ylabel('TXPRGDIVCLK (MHz)')
ax1.grid(True, alpha=0.3)
ax1.set_title('Variazione frequenza QPLL0 con SDM_DATA')

ax2.plot(sdm_used, freqs_vco, 's-', color='darkorange')
ax2.set_ylabel('VCO stima (GHz)')
ax2.set_xlabel('SDM_DATA [22-bit]')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

if len(freqs_vco) > 1:
    span_mhz = (max(freqs_vco) - min(freqs_vco)) * 1000
    print(f'\nSpan VCO osservato: {span_mhz:.3f} MHz su {N_STEPS} passi')
    print(f'Step medio: {span_mhz / (N_STEPS - 1):.4f} MHz/step')

In [ ]:
# --- Monitor live (aggiornamento continuo ogni finestra) ---
import ipywidgets as widgets
from IPython.display import display, clear_output

out = widgets.Output()
stop_btn = widgets.Button(description='Stop', button_style='danger')
running = [True]

def on_stop(_):
    running[0] = False
    print('Monitor fermato.')

stop_btn.on_click(on_stop)
display(stop_btn, out)

history_tx  = []
history_time = []
t0 = time.time()

while running[0]:
    try:
        ft, fv = read_frequency(mm_fc, timeout_s=gate_seconds + 3)
        t_now = time.time() - t0
        history_tx.append(ft / 1e6)
        history_time.append(t_now)
        # Mantieni ultimi 120 campioni
        if len(history_tx) > 120:
            history_tx.pop(0)
            history_time.pop(0)
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 3))
            ax.plot(history_time, history_tx, '-', color='steelblue')
            ax.set_xlabel('Tempo (s)')
            ax.set_ylabel('TXPRGDIVCLK (MHz)')
            ax.set_title(f'Ultimo: {ft/1e6:.4f} MHz  |  VCO≈{fv/1e9:.6f} GHz')
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
    except TimeoutError:
        with out:
            clear_output(wait=True)
            print('Attendo risultato dal frequenzimetro...')
    except KeyboardInterrupt:
        break

In [ ]:
# --- Cleanup: chiudi mmap ---
mm_fc.close();   f_fc.close()
mm_comm.close(); f_comm.close()
print('Risorse rilasciate.')